# 03 - LSTM and GRU: Why They Work

---

## Learning Objectives

By the end of this notebook, you will be able to:

- Describe the LSTM architecture: **cell state** and three gates (forget, input, output) with their full equations
- Describe the GRU architecture: **update gate** and **reset gate** as a simplified alternative to LSTM
- Explain **why gating mechanisms solve the vanishing gradient problem**
- Use `nn.LSTM` and `nn.GRU` in PyTorch to process sequences
- Compare RNN vs LSTM vs GRU on a **long-range dependency** task

## Prerequisites

- Notebook 02: RNN From Scratch (RNN cell equation, vanishing gradients)
- Understanding of sigmoid $\sigma$ and tanh activations
- Element-wise (Hadamard) product notation: $\odot$

## Table of Contents

1. [Recap: Why Vanilla RNNs Struggle](#1-recap-why-vanilla-rnns-struggle)
2. [LSTM: Long Short-Term Memory](#2-lstm-long-short-term-memory)
3. [LSTM Gate Equations (Full LaTeX)](#3-lstm-gate-equations-full-latex)
4. [Why Gates Solve Vanishing Gradients](#4-why-gates-solve-vanishing-gradients)
5. [GRU: Gated Recurrent Unit](#5-gru-gated-recurrent-unit)
6. [LSTM vs GRU: When to Use Which](#6-lstm-vs-gru-when-to-use-which)
7. [Code: nn.LSTM and nn.GRU on Synthetic Sequences](#7-code-nnlstm-and-nngru-on-synthetic-sequences)
8. [Code: RNN vs LSTM vs GRU on Long-Range Dependency](#8-code-rnn-vs-lstm-vs-gru-on-long-range-dependency)
9. [Exercise: Experiment with Hidden Sizes](#9-exercise-experiment-with-hidden-sizes)
10. [Common Mistakes & Debugging Tips](#10-common-mistakes--debugging-tips)

In [ ]:
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

set_global_seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100

---
## 1. Recap: Why Vanilla RNNs Struggle

From notebook 02, we saw that the vanilla RNN:

$$h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1} + b)$$

suffers from the **vanishing gradient problem**:

- Gradients flow through a product of $W_{hh}$ and $\tanh'$ at each time step
- Over many steps, gradients either **vanish** (if $\|W_{hh}\| < 1$) or **explode** (if $\|W_{hh}\| > 1$)
- The network cannot learn dependencies spanning more than ~10--20 time steps

### The core issue

In a vanilla RNN, information is **always transformed** by a nonlinearity at every step. There is no mechanism to **preserve** information unchanged across time.

### The solution: gates

LSTMs and GRUs introduce **gating mechanisms** that can:
- **Preserve** information by passing it through unchanged (gate $\approx 1$)
- **Forget** information when it is no longer relevant (gate $\approx 0$)
- **Add** new information selectively

---
## 2. LSTM: Long Short-Term Memory

The LSTM (Hochreiter & Schmidhuber, 1997) introduces two key innovations:

### 1. Cell state $C_t$ (the "conveyor belt")

- A separate memory vector that runs through time **with only linear interactions**
- Information can flow along $C_t$ unchanged (like a conveyor belt)
- This is the key mechanism that enables long-range memory

### 2. Three gates (all using sigmoid $\sigma$ to produce values in $[0, 1]$)

| Gate | Symbol | Purpose |
|------|--------|---------|
| **Forget gate** | $f_t$ | Decides what to **remove** from cell state |
| **Input gate** | $i_t$ | Decides what **new information** to store |
| **Output gate** | $o_t$ | Decides what to **output** from cell state |

Each gate is a learned sigmoid layer that outputs values between 0 ("block everything") and 1 ("let everything through").

---
## 3. LSTM Gate Equations (Full LaTeX)

At each time step $t$, given input $x_t$, previous hidden state $h_{t-1}$, and previous cell state $C_{t-1}$:

### Forget gate
$$f_t = \sigma(W_f[h_{t-1}, x_t] + b_f)$$

Decides which components of $C_{t-1}$ to keep. Values near 0 mean "forget", near 1 mean "remember".

### Input gate + candidate cell state
$$i_t = \sigma(W_i[h_{t-1}, x_t] + b_i)$$
$$\tilde{C}_t = \tanh(W_C[h_{t-1}, x_t] + b_C)$$

The input gate $i_t$ controls how much of the new candidate $\tilde{C}_t$ to add.

### Cell state update
$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$

This is the crucial equation: the cell state is a **linear combination** of the old state and new candidate, controlled by gates.

### Output gate + hidden state
$$o_t = \sigma(W_o[h_{t-1}, x_t] + b_o)$$
$$h_t = o_t \odot \tanh(C_t)$$

The output gate controls which parts of the cell state are exposed as the hidden state.

### Notation
- $[h_{t-1}, x_t]$ denotes concatenation of $h_{t-1}$ and $x_t$
- $\odot$ denotes element-wise (Hadamard) product
- $\sigma$ is the sigmoid function: $\sigma(z) = \frac{1}{1 + e^{-z}}$

In [ ]:
# Visualize LSTM gate behavior with different inputs
set_global_seed(42)

# Show sigmoid (gate activation) and tanh (candidate/output)
z = torch.linspace(-6, 6, 200)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Sigmoid (gate)
axes[0].plot(z.numpy(), torch.sigmoid(z).numpy(), 'b-', linewidth=2)
axes[0].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
axes[0].fill_between(z.numpy(), 0, torch.sigmoid(z).numpy(), alpha=0.1)
axes[0].set_title('Sigmoid $\\sigma(z)$: Gate Activation', fontweight='bold')
axes[0].set_xlabel('z')
axes[0].set_ylabel('$\\sigma(z)$')
axes[0].annotate('"Forget" (gate $\\approx$ 0)', xy=(-4, 0.02), fontsize=11, color='red')
axes[0].annotate('"Remember" (gate $\\approx$ 1)', xy=(1.5, 0.9), fontsize=11, color='green')
axes[0].grid(True, alpha=0.3)

# Tanh (candidate)
axes[1].plot(z.numpy(), torch.tanh(z).numpy(), 'r-', linewidth=2)
axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[1].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_title('Tanh: Candidate / Output Activation', fontweight='bold')
axes[1].set_xlabel('z')
axes[1].set_ylabel('tanh(z)')
axes[1].set_ylim(-1.3, 1.3)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Gates use sigmoid (0 to 1): they control information flow.")
print("Candidates use tanh (-1 to 1): they produce bounded values to store.")

---
## 4. Why Gates Solve Vanishing Gradients

The cell state update equation is the key:

$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$

### Gradient through the cell state

$$\frac{\partial C_t}{\partial C_{t-1}} = \text{diag}(f_t) + \text{(terms involving gates)}$$

Over $T$ time steps:

$$\frac{\partial C_T}{\partial C_k} = \prod_{t=k+1}^{T} f_t \quad (\text{approximately, for the direct path})$$

### Why this helps

- When $f_t \approx 1$ (forget gate open), the gradient passes through **undiminished**
- The network **learns** when to keep the gate open (preserve memory) vs closed (forget)
- Compared to vanilla RNN where $\frac{\partial h_t}{\partial h_{t-1}} = \text{diag}(1 - h_t^2) \cdot W_{hh}$, the LSTM avoids the repeated multiplication by $W_{hh}$
- This is analogous to how **residual connections** in ResNets help gradient flow in deep networks

In [ ]:
# Demonstrate: LSTM maintains gradient magnitude over long sequences
set_global_seed(42)

seq_len = 50
input_size = 10
hidden_size = 32

# RNN gradient norms (from notebook 02 concept)
rnn = nn.RNN(input_size, hidden_size, batch_first=True)
lstm = nn.LSTM(input_size, hidden_size, batch_first=True)

x = torch.randn(1, seq_len, input_size, requires_grad=True)

# RNN: compute gradient of output w.r.t. each time step's input
rnn_out, _ = rnn(x)
rnn_loss = rnn_out[0, -1, :].sum()  # loss from final step
rnn_loss.backward()
rnn_grad_norms = [x.grad[0, t, :].norm().item() for t in range(seq_len)]

# LSTM
x2 = torch.randn(1, seq_len, input_size, requires_grad=True)
lstm_out, _ = lstm(x2)
lstm_loss = lstm_out[0, -1, :].sum()
lstm_loss.backward()
lstm_grad_norms = [x2.grad[0, t, :].norm().item() for t in range(seq_len)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].semilogy(range(seq_len), rnn_grad_norms, 'r-o', markersize=3, label='RNN')
axes[0].semilogy(range(seq_len), lstm_grad_norms, 'b-s', markersize=3, label='LSTM')
axes[0].set_xlabel('Time Step')
axes[0].set_ylabel('Gradient Norm (log scale)')
axes[0].set_title('Gradient Norms: RNN vs LSTM', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Ratio plot
rnn_ratio = [rnn_grad_norms[t] / (rnn_grad_norms[-1] + 1e-10) for t in range(seq_len)]
lstm_ratio = [lstm_grad_norms[t] / (lstm_grad_norms[-1] + 1e-10) for t in range(seq_len)]

axes[1].semilogy(range(seq_len), rnn_ratio, 'r-o', markersize=3, label='RNN')
axes[1].semilogy(range(seq_len), lstm_ratio, 'b-s', markersize=3, label='LSTM')
axes[1].set_xlabel('Time Step')
axes[1].set_ylabel('Relative Gradient Norm (log scale)')
axes[1].set_title('Relative Gradient: Normalized by Final Step', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"RNN  gradient ratio (first/last): {rnn_ratio[0]:.6f}")
print(f"LSTM gradient ratio (first/last): {lstm_ratio[0]:.6f}")
print("\nLSTM maintains much stronger gradients at early time steps.")

---
## 5. GRU: Gated Recurrent Unit

The GRU (Cho et al., 2014) is a simplified version of LSTM with:
- **No separate cell state** -- only the hidden state $h_t$
- **Two gates** instead of three (fewer parameters)

### GRU Equations

**Update gate** (combines LSTM's forget and input gates):
$$z_t = \sigma(W_z[h_{t-1}, x_t] + b_z)$$

**Reset gate** (controls how much of previous state to use for candidate):
$$r_t = \sigma(W_r[h_{t-1}, x_t] + b_r)$$

**Candidate hidden state:**
$$\tilde{h}_t = \tanh(W_h[r_t \odot h_{t-1}, x_t] + b_h)$$

**Hidden state update** (interpolation between old and new):
$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$$

### Key insight

When $z_t \approx 0$: $h_t \approx h_{t-1}$ (carry information forward unchanged)

When $z_t \approx 1$: $h_t \approx \tilde{h}_t$ (completely replace with new candidate)

---
## 6. LSTM vs GRU: When to Use Which

| Feature | LSTM | GRU |
|---------|------|-----|
| Hidden vectors | $h_t$ + $C_t$ (two states) | $h_t$ only |
| Gates | 3 (forget, input, output) | 2 (update, reset) |
| Parameters | More ($4 \times$ hidden layers) | Fewer ($3 \times$ hidden layers) |
| Training speed | Slower | Faster |
| Memory usage | Higher | Lower |
| Long sequences | Slightly better in theory | Comparable in practice |
| Default choice | Large datasets, complex tasks | Smaller datasets, faster iteration |

### Rules of thumb

- **Start with GRU** if you want faster training and have limited compute
- **Use LSTM** when you need maximum expressiveness or have very long sequences
- In practice, performance is often **comparable** -- experiment with both

---
## 7. Code: nn.LSTM and nn.GRU on Synthetic Sequences

Let us use PyTorch's built-in LSTM and GRU to process sequences and inspect their outputs.

In [ ]:
set_global_seed(42)

# Common settings
batch_size = 2
seq_len = 10
input_size = 8
hidden_size = 16

x = torch.randn(batch_size, seq_len, input_size)

# --- LSTM ---
lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
lstm_out, (h_n, c_n) = lstm(x)

print("=== LSTM ===")
print(f"Input shape:        {x.shape}")
print(f"Output (all steps): {lstm_out.shape}  (batch, seq_len, hidden_size)")
print(f"h_n (final hidden): {h_n.shape}      (num_layers*directions, batch, hidden_size)")
print(f"c_n (final cell):   {c_n.shape}      (num_layers*directions, batch, hidden_size)")
print(f"Last output == h_n: {torch.allclose(lstm_out[:, -1, :], h_n.squeeze(0), atol=1e-6)}")
print()

In [ ]:
set_global_seed(42)

# --- GRU ---
gru = nn.GRU(input_size, hidden_size, batch_first=True)
gru_out, h_n_gru = gru(x)

print("=== GRU ===")
print(f"Input shape:        {x.shape}")
print(f"Output (all steps): {gru_out.shape}  (batch, seq_len, hidden_size)")
print(f"h_n (final hidden): {h_n_gru.shape}  (num_layers*directions, batch, hidden_size)")
print(f"Last output == h_n: {torch.allclose(gru_out[:, -1, :], h_n_gru.squeeze(0), atol=1e-6)}")
print()
print("Note: GRU returns only h_n (no separate cell state c_n).")

In [ ]:
# Compare parameter counts
rnn_layer = nn.RNN(input_size, hidden_size, batch_first=True)
lstm_layer = nn.LSTM(input_size, hidden_size, batch_first=True)
gru_layer = nn.GRU(input_size, hidden_size, batch_first=True)

def count_params(model):
    return sum(p.numel() for p in model.parameters())

print(f"Parameter counts (input_size={input_size}, hidden_size={hidden_size}):")
print(f"  RNN:  {count_params(rnn_layer):,}")
print(f"  LSTM: {count_params(lstm_layer):,}  ({count_params(lstm_layer)/count_params(rnn_layer):.1f}x RNN)")
print(f"  GRU:  {count_params(gru_layer):,}  ({count_params(gru_layer)/count_params(rnn_layer):.1f}x RNN)")
print()
print("LSTM has ~4x RNN parameters (4 gate/candidate weight sets).")
print("GRU has ~3x RNN parameters (3 gate/candidate weight sets).")

---
## 8. Code: RNN vs LSTM vs GRU on Long-Range Dependency

We create a synthetic task where the model must remember a signal from the **first** time step to make a prediction at the **last** time step. This directly tests long-range memory.

### Task description

- Input: sequence of length $T$ filled with random noise
- The **first element** is set to either +1 or -1 (the "signal")
- Target: classify whether the first element was +1 or -1
- The model must remember across $T$ steps of noise

In [ ]:
set_global_seed(42)

def generate_long_range_data(n_samples, seq_len, noise_std=0.1):
    """Generate sequences where the first element determines the class.
    
    Returns:
        X: (n_samples, seq_len, 1) input sequences
        y: (n_samples,) binary labels
    """
    X = torch.randn(n_samples, seq_len, 1) * noise_std
    y = torch.randint(0, 2, (n_samples,))  # 0 or 1
    # Set first time step to +1 or -1 based on label
    X[:, 0, 0] = y.float() * 2 - 1  # map 0->-1, 1->+1
    return X, y

# Generate data with long sequences
SEQ_LEN = 50
N_TRAIN = 1000
N_TEST = 200

X_train, y_train = generate_long_range_data(N_TRAIN, SEQ_LEN)
X_test, y_test = generate_long_range_data(N_TEST, SEQ_LEN)

print(f"Training data: X={X_train.shape}, y={y_train.shape}")
print(f"Test data:     X={X_test.shape}, y={y_test.shape}")
print(f"\nSequence length: {SEQ_LEN} (signal at step 0, predict at step {SEQ_LEN-1})")
print(f"Class balance: {y_train.float().mean():.2f} (should be ~0.5)")
print(f"\nExample first 5 values of sequence 0: {X_train[0, :5, 0].tolist()}")
print(f"Label: {y_train[0].item()} (first element: {X_train[0, 0, 0].item():.1f})")

In [ ]:
class SequenceClassifier(nn.Module):
    """Wraps an RNN/LSTM/GRU for binary sequence classification."""
    
    def __init__(self, rnn_type, input_size, hidden_size):
        super().__init__()
        if rnn_type == 'RNN':
            self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        elif rnn_type == 'LSTM':
            self.rnn = nn.LSTM(input_size, hidden_size, batch_first=True)
        elif rnn_type == 'GRU':
            self.rnn = nn.GRU(input_size, hidden_size, batch_first=True)
        else:
            raise ValueError(f"Unknown rnn_type: {rnn_type}")
        
        self.rnn_type = rnn_type
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        # x: (batch, seq_len, input_size)
        output, hidden = self.rnn(x)
        # Use the final hidden state for classification
        last_output = output[:, -1, :]  # (batch, hidden_size)
        logit = self.fc(last_output).squeeze(-1)  # (batch,)
        return logit


def train_and_evaluate(rnn_type, X_train, y_train, X_test, y_test,
                       hidden_size=32, n_epochs=100, lr=0.001):
    """Train a model and return train/test accuracy history."""
    set_global_seed(42)
    
    model = SequenceClassifier(rnn_type, input_size=1, hidden_size=hidden_size)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    
    # Simple batch training (dataset is small enough)
    train_accs = []
    test_accs = []
    losses = []
    
    for epoch in range(n_epochs):
        model.train()
        logits = model(X_train)
        loss = loss_fn(logits, y_train.float())
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        
        losses.append(loss.item())
        
        # Evaluate every 10 epochs
        if (epoch + 1) % 10 == 0:
            model.eval()
            with torch.no_grad():
                train_preds = (torch.sigmoid(model(X_train)) > 0.5).long()
                test_preds = (torch.sigmoid(model(X_test)) > 0.5).long()
                train_acc = (train_preds == y_train).float().mean().item()
                test_acc = (test_preds == y_test).float().mean().item()
            train_accs.append(train_acc)
            test_accs.append(test_acc)
    
    return losses, train_accs, test_accs

print("Training three models... (this may take a moment)")

In [ ]:
# Train all three models
results = {}
for rnn_type in ['RNN', 'LSTM', 'GRU']:
    losses, train_accs, test_accs = train_and_evaluate(
        rnn_type, X_train, y_train, X_test, y_test,
        hidden_size=32, n_epochs=150, lr=0.001
    )
    results[rnn_type] = {
        'losses': losses,
        'train_accs': train_accs,
        'test_accs': test_accs
    }
    print(f"{rnn_type:>4s} | Final train acc: {train_accs[-1]:.3f} | Final test acc: {test_accs[-1]:.3f}")

In [ ]:
# Plot results
colors = {'RNN': '#e74c3c', 'LSTM': '#3498db', 'GRU': '#2ecc71'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training loss
for rnn_type, res in results.items():
    axes[0].plot(res['losses'], color=colors[rnn_type], label=rnn_type, linewidth=1.5, alpha=0.8)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title(f'Training Loss (seq_len={SEQ_LEN})', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Test accuracy
eval_epochs = list(range(10, 151, 10))
for rnn_type, res in results.items():
    axes[1].plot(eval_epochs, res['test_accs'], '-o', color=colors[rnn_type],
                 label=rnn_type, linewidth=1.5, markersize=4)
axes[1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random baseline')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Test Accuracy')
axes[1].set_title(f'Test Accuracy (seq_len={SEQ_LEN})', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0.4, 1.05)

plt.tight_layout()
plt.show()

print(f"\nWith sequence length {SEQ_LEN}:")
print(f"  RNN struggles to learn the long-range dependency (vanishing gradients).")
print(f"  LSTM and GRU can solve the task because their gates preserve the signal.")

---
## 9. Exercise: Experiment with Hidden Sizes

**Task:**

1. Vary the hidden size in `{8, 16, 32, 64}` for LSTM on the long-range dependency task
2. For each hidden size, train for 150 epochs and record the final test accuracy
3. Plot hidden size vs test accuracy
4. Answer: Does a larger hidden size always help? What is the minimum hidden size that works?

```python
# Your code here
set_global_seed(42)

# hidden_sizes = [8, 16, 32, 64]
# for hs in hidden_sizes:
#     losses, train_accs, test_accs = train_and_evaluate(
#         'LSTM', X_train, y_train, X_test, y_test,
#         hidden_size=hs, n_epochs=150
#     )
#     print(f"  hidden_size={hs:3d} | test acc = {test_accs[-1]:.3f}")
```

In [ ]:
# Try the exercise yourself before looking at the solution!





### Solution

In [ ]:
# ----- Solution -----
set_global_seed(42)

hidden_sizes = [8, 16, 32, 64]
hs_results = {}

for hs in hidden_sizes:
    losses, train_accs, test_accs = train_and_evaluate(
        'LSTM', X_train, y_train, X_test, y_test,
        hidden_size=hs, n_epochs=150, lr=0.001
    )
    hs_results[hs] = test_accs[-1]
    n_params = sum(p.numel() for p in SequenceClassifier('LSTM', 1, hs).parameters())
    print(f"  hidden_size={hs:3d} | test acc = {test_accs[-1]:.3f} | params = {n_params:,}")

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar([str(h) for h in hidden_sizes],
       [hs_results[h] for h in hidden_sizes],
       color='#3498db', edgecolor='black', alpha=0.8)
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Random baseline')
ax.set_xlabel('Hidden Size')
ax.set_ylabel('Test Accuracy')
ax.set_title('LSTM Test Accuracy vs Hidden Size', fontweight='bold')
ax.set_ylim(0.4, 1.05)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\nObservation: Even small hidden sizes can solve this task,")
print("because the gate mechanism (not the hidden size) is what enables")
print("long-range memory. Larger hidden size helps with more complex patterns.")

---
## 10. Common Mistakes & Debugging Tips

**1. Forgetting that LSTM returns `(output, (h_n, c_n))` while GRU returns `(output, h_n)`**
- LSTM has a separate cell state `c_n`; GRU does not.
- Unpacking incorrectly causes shape errors.

**2. Not applying gradient clipping with RNNs**
- Always use `torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)` during training.
- Even LSTMs/GRUs can experience gradient spikes, especially early in training.

**3. Confusing `output` and `h_n` shapes**
- `output`: hidden state at **every** time step -- shape `(batch, seq_len, hidden_size)`
- `h_n`: hidden state at the **last** time step -- shape `(num_layers * directions, batch, hidden_size)`
- For classification: use `output[:, -1, :]` or `h_n.squeeze(0)` (they are the same for single-layer unidirectional).

**4. Forgetting `batch_first=True`**
- Default PyTorch RNN/LSTM/GRU expects `(seq_len, batch, features)`. Most people prefer `(batch, seq_len, features)` -- set `batch_first=True`.

**5. Using too many stacked layers without residual connections**
- Stacking 2--3 layers is common. Beyond that, consider adding residual/skip connections or using Transformers.

**6. Not initializing hidden state for multi-layer/bidirectional models**
- For `num_layers=2, bidirectional=True`, the initial hidden state shape must be `(2*2, batch, hidden_size)` -- not `(1, batch, hidden_size)`.
- When in doubt, let PyTorch initialize it to zeros by not passing `h_0`.

---

**Next notebook:** [04 - Text Classification with RNN/LSTM](04_Text_Classification_with_RNN_LSTM.ipynb) -- we apply LSTM to a real-world text classification task end to end.